In [ ]:
import requests
import os
import urllib.request # Tá sendo usado: bug da IDE
import csv
import io

import boto3
import json
import re

from botocore.exceptions import ClientError
from abc import ABC, abstractmethod
from typing import List, Dict
from datetime import datetime, timezone
from zoneinfo import ZoneInfo


import pandas as pd

# Criação de Classes

## S3

In [31]:
class S3Access:
    def __init__(self):
        self.s3 = boto3.client("s3")
        self.bucket = "projeto-mestrado-renan-gomes"

    def object_exists(self, key: str) -> bool:
        try:
            self.s3.head_object(Bucket=self.bucket, Key=key)
            return True
        except ClientError as e:
            if e.response["Error"]["Code"] in ("404", "NoSuchKey", "NotFound"):
                return False
            raise


    def ler_dataframe(self, s3_path: str, delimiter: str = ",", encoding: str = "utf-8") -> pd.DataFrame:
        """
        Lê um CSV do S3 e retorna um DataFrame.
        Retorna DataFrame vazio se o arquivo não existir.
        """
        if not self.object_exists(s3_path):
            return pd.DataFrame()

        body = self.s3.get_object(Bucket=self.bucket, Key=s3_path)["Body"].read().decode(encoding)
        return pd.read_csv(io.StringIO(body), sep=delimiter)


    def adicionar_dataframe(
        self,
        s3_path: str,
        df: pd.DataFrame,
        delimiter: str = ",",
        encoding: str = "utf-8",
    ) -> pd.DataFrame:
        """
        Adiciona as linhas de `df` ao final do CSV no S3. Se o arquivo não
        existir, cria um novo com as colunas de `df`.

        Colunas novas em `df` são acrescentadas ao final do cabeçalho; colunas
        que faltarem ficam vazias. O S3 não suporta append, então o arquivo é
        lido, concatenado e regravado inteiro.

        Retorna o DataFrame completo que foi gravado.
        """
        existente = self.ler_dataframe(s3_path, delimiter=delimiter, encoding=encoding)

        if existente.empty:
            final = df.reset_index(drop=True)
        else:
            final = pd.concat([existente, df], ignore_index=True, sort=False)

        buffer = io.StringIO()
        final.to_csv(buffer, sep=delimiter, index=False)

        self.s3.put_object(
            Bucket=self.bucket,
            Key=s3_path,
            Body=buffer.getvalue().encode(encoding),
            ContentType="text/csv",
        )
        return final


## APIs

### Constantes

In [32]:
FATORES_EMISSAO = {
    "carvao": 820,
    "gas": 490,
    "oleo": 650,             # suposição
    "biomassa": 230,
    "solar": 48,
    "geotermica": 38,
    "hidraulica": 24,
    "nuclear": 12,
    "eolica": 11,
    "termica_agregada": 490, # suposição: térmica sem separação (ex.: ONS); usei o fator do gás
    "outros": 700,           # suposição: "Other/Others/Waste" sem detalhamento
}

RENOVAVEIS = {"hidraulica", "eolica", "solar", "biomassa", "geotermica"}
LIVRE_CARBONO = RENOVAVEIS | {"nuclear"}

### Utils

In [33]:
def baixar_json(url):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=30) as resp:
        return json.loads(resp.read().decode("utf-8"))
def calcular(mix_mw):
    """mix_mw: {fonte: MW}. Negativos viram 0."""
    mix = {k: max(v or 0, 0) for k, v in mix_mw.items()}
    total = sum(mix.values())
    if total <= 0:
        raise ValueError("geração total zero")
    emissoes = sum(v * FATORES_EMISSAO[k] for k, v in mix.items())
    return {
        "total_mw": round(total, 2),
        "pct_renovavel": round(100 * sum(v for k, v in mix.items() if k in RENOVAVEIS) / total, 2),
        "pct_livre_carbono": round(100 * sum(v for k, v in mix.items() if k in LIVRE_CARBONO) / total, 2),
        "intensidade_gco2_kwh": round(emissoes / total, 1),
        "dados_brutos": {k: round(v, 2) for k, v in mix.items() if v > 0},
    }

def imprimir(regiao, horario, r):
    print(f"{regiao:22} {horario}  "
          f"{r['intensidade_gco2_kwh']:>6} gCO2eq/kWh  "
          f"renovável {r['pct_renovavel']:6.2f}%  "
          f"livre de carbono {r['pct_livre_carbono']:6.2f}%")



### Classes

In [34]:
class APIWrapper(ABC):
    @abstractmethod
    def request() -> List[str]: # Sempre retorna: Nome, Horário e o Mix
        ...
    def print_request(self) -> List[str]:
        req = self.request()
        for nome, horario, r in req:
            imprimir(nome, horario, r) 
        return req

In [35]:
class Regiao_BR_CS(APIWrapper):
    URL = "https://tr.ons.org.br/Content/GetBalancoEnergetico/null"
    SUBSISTEMAS = {
        "sudesteECentroOeste": "Brasil SE/CO",
        "sul": "Brasil S",
        "nordeste": "Brasil NE",
        "norte": "Brasil N",
    }
    CAMPOS = {
        "hidraulica": "hidraulica",
        "itaipu50HzBrasil": "hidraulica",
        "itaipu60Hz": "hidraulica",
        "termica": "termica_agregada",
        "eolica": "eolica",
        "nuclear": "nuclear",
        "solar": "solar",
        "mmgd": "solar",
    }

    @classmethod
    def request(cls) -> List[str]:
        dados = baixar_json(cls.URL)
        resultados = []
        for chave, nome in cls.SUBSISTEMAS.items():
            mix = {}
            for campo, valor in dados[chave]["geracao"].items():
                fonte = cls.CAMPOS.get(campo)
                if fonte:
                    mix[fonte] = mix.get(fonte, 0) + max(valor or 0, 0)
            resultados.append((nome, dados["Data"], calcular(mix)))
        return resultados
# RegiaoBRCS().request()

In [36]:
class Regiao_US_CAL_CISO(APIWrapper):
    URL = "https://www.caiso.com/outlook/current/fuelsource.csv"
    COLUNAS = {
        "Solar": "solar",
        "Wind": "eolica",
        "Geothermal": "geotermica",
        "Biomass": "biomassa",
        "Biogas": "biomassa",
        "Small hydro": "hidraulica",
        "Large Hydro": "hidraulica",
        "Coal": "carvao",
        "Nuclear": "nuclear",
        "Natural Gas": "gas",
        "Other": "outros",
    }
    @classmethod
    def request(cls)-> List[str]:
        req = urllib.request.Request(cls.URL, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=30) as resp:
            texto = resp.read().decode("utf-8")

        linhas = list(csv.DictReader(io.StringIO(texto)))
        # última linha com todas as colunas preenchidas
        completas = [l for l in linhas if all((l.get(c) or "").strip() for c in cls.COLUNAS)]
        if not completas:
            raise ValueError("CSV sem linhas completas")
        ultima = completas[-1]

        mix = {}
        for coluna, fonte in cls.COLUNAS.items():
            mix[fonte] = mix.get(fonte, 0) + float(ultima[coluna])
        return [("EUA Califórnia", f"hoje {ultima['Time']} (horário do Pacífico)", calcular(mix))]

In [37]:
class Regiao_FR(APIWrapper):
    URL = ("https://odre.opendatasoft.com/api/explore/v2.1/catalog/datasets/"
        "eco2mix-national-tr/records?order_by=date_heure%20desc&limit=100")

    CAMPOS = {
        "fioul": "oleo",
        "charbon": "carvao",
        "gaz": "gas",
        "nucleaire": "nuclear",
        "eolien": "eolica",
        "solaire": "solar",
        "hydraulique": "hidraulica",
        "bioenergies": "biomassa",
    }

    @classmethod
    def request(cls)->List[str]:
        dados = baixar_json(cls.URL)
        medidos = [r for r in dados["results"] if r.get("nucleaire") is not None]
        if not medidos:
            raise ValueError("nenhum registro com dado medido nos últimos 100")
        reg = medidos[0]  # já vem ordenado do mais recente para o mais antigo

        mix = {}
        for campo, fonte in cls.CAMPOS.items():
            mix[fonte] = mix.get(fonte, 0) + (reg.get(campo) or 0)

        r = calcular(mix)
        r["taux_co2_rte"] = reg.get("taux_co2")
        return [("França", reg["date_heure"], r)]


In [38]:
class Regiao_GB(APIWrapper):
    URL = "https://api.carbonintensity.org.uk/generation"
    COMBUSTIVEIS = {
        "biomass": "biomassa",
        "coal": "carvao",
        "gas": "gas",
        "nuclear": "nuclear",
        "other": "outros",
        "hydro": "hidraulica",
        "solar": "solar",
        "wind": "eolica",
    }

    @classmethod
    def request(cls)->List[str]:
        dados = baixar_json(cls.URL)["data"]
        mix = {}
        for item in dados["generationmix"]:
            fonte = cls.COMBUSTIVEIS.get(item["fuel"])
            if fonte:  # imports e desconhecidos ficam de fora
                mix[fonte] = mix.get(fonte, 0) + item["perc"]
        return [("Reino Unido", dados["from"], calcular(mix))]

In [39]:
class Regiao_NZ(APIWrapper):
    URL = "https://www.transpower.co.nz/power-system-live-data"
    FONTES = {
        "coal": "carvao",
        "gas": "gas",
        "geothermal": "geotermica",
        "hydro": "hidraulica",
        "wind": "eolica",
        "solar": "solar",
        "co-gen": "outros",
        "cogen": "outros",
        "liquid": "oleo",
        "diesel": "oleo",
        "diesel/oil": "oleo",
    }
    IGNORAR = {"battery"}

    @classmethod
    def __objeto_em_volta(cls,texto, pos):
        """Retorna o objeto JSON {...} que contém a posição pos."""
        nivel, ini = 0, pos
        while ini >= 0:
            if texto[ini] == "}":
                nivel += 1
            elif texto[ini] == "{":
                if nivel == 0:
                    break
                nivel -= 1
            ini -= 1
        nivel = 0
        for fim in range(ini, len(texto)):
            if texto[fim] == "{":
                nivel += 1
            elif texto[fim] == "}":
                nivel -= 1
                if nivel == 0:
                    return json.loads(texto[ini:fim + 1])
        raise ValueError("objeto JSON não fechado")


    @classmethod
    def request(cls)-> List[str]:
        req = urllib.request.Request(cls.URL, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=30) as resp:
            html = resp.read().decode("utf-8")

        candidatos = []
        for m in re.finditer(r'"Coal"\s*:\s*\{\s*"generation"', html):
            try:
                candidatos.append(cls.__objeto_em_volta(html, m.start()))
            except (ValueError, json.JSONDecodeError):
                continue
        if not candidatos:
            raise ValueError("não achei o JSON de geração na página")

        def total(obj):
            return sum(max(v.get("generation") or 0, 0)
                    for v in obj.values() if isinstance(v, dict))

        nacional = max(candidatos, key=total)

        mix, nao_mapeados = {}, []
        for nome, v in nacional.items():
            if not isinstance(v, dict) or "generation" not in v:
                continue
            chave = nome.lower()
            if chave in cls.IGNORAR:
                continue
            fonte = cls.FONTES.get(chave)
            if not fonte:
                nao_mapeados.append(nome)
                continue
            mix[fonte] = mix.get(fonte, 0) + (v["generation"] or 0)

        if nao_mapeados:
            print(f"[NZ] fontes não reconhecidas (ignoradas): {nao_mapeados}")

        m = re.search(r"as at.{0,120}?(\d{1,2} \w{3} \d{4} \d{1,2}:\d{2})", html, re.S | re.I)
        horario = m.group(1) + " (horário NZ)" if m else "horário não encontrado na página"
        return [("Nova Zelândia", horario, calcular(mix))]

In [40]:
class Regiao_EURO(APIWrapper):
    URL = "https://api.energy-charts.info/public_power?country={pais}"
    IGNORAR = ("consumption", "cross border", "load", "share", "battery")
    PALAVRAS = [
        ("nuclear", "nuclear"),
        ("lignite", "carvao"),
        ("hard coal", "carvao"),
        ("oil", "oleo"),
        ("gas", "gas"),
        ("biomass", "biomassa"),
        ("geothermal", "geotermica"),
        ("hydro", "hidraulica"),
        ("wind", "eolica"),
        ("solar", "solar"),
        ("waste", "outros"),
        ("other", "outros"),
    ]
    def __init__(self, pais, nome_regiao):
        self.pais = pais
        self.nome_regiao = nome_regiao
        super().__init__()

    @classmethod
    def classificar(cls,nome):
        n = nome.lower()
        if any(p in n for p in cls.IGNORAR):
            return None
        for palavra, fonte in cls.PALAVRAS:
            if palavra in n:
                return fonte
        return "?"

    def request(self) -> List[str]:
        dados = baixar_json(self.URL.format(pais=self.pais))
        tempos = dados["unix_seconds"]
        tipos = dados["production_types"]

        # instante mais recente com o maior número de séries preenchidas
        # (algumas séries podem vir sempre vazias, então não exige todas)
        relevantes = [t for t in tipos if self.classificar(t["name"])]
        preenchidas = [sum(t["data"][j] is not None for t in relevantes) for j in range(len(tempos))]
        if not preenchidas or max(preenchidas) == 0:
            raise ValueError("nenhum instante com dados")
        melhor = max(preenchidas)
        i = max(j for j, n in enumerate(preenchidas) if n == melhor)

        mix, nao_mapeados = {}, []
        for t in tipos:
            fonte = self.classificar(t["name"])
            if fonte is None:
                continue
            if fonte == "?":
                nao_mapeados.append(t["name"])
                continue
            mix[fonte] = mix.get(fonte, 0) + (t["data"][i] or 0)

        if nao_mapeados:
            print(f"[{self.pais.upper()}] fontes não reconhecidas (ignoradas): {nao_mapeados}")

        horario = datetime.fromtimestamp(tempos[i], tz=timezone.utc).isoformat()
        return [(self.nome_regiao, horario, calcular(mix))]


In [41]:
regioes: Dict[str,APIWrapper] = {
    "BR_CS":Regiao_BR_CS(),                      # 1
    "US_CAL_CISO":Regiao_US_CAL_CISO(),          # 2
    "FR":Regiao_FR(),                            # 3
    "Alemanha":Regiao_EURO("de", "Alemanha"),    # 4
    "GB":Regiao_GB(),                            # 5
    "NZ":Regiao_NZ(),                            # 6
    "PL":Regiao_EURO("pl", "Polônia"),           # 7
    "ES":Regiao_EURO("es", "Espanha"),           # 8
    "IT":Regiao_EURO("it", "Itália"),            # 9
    "SE":Regiao_EURO("se", "Suécia"),            # 10
}

In [42]:
erros = 0
data = []

for nome,obj in regioes.items():
    print("---------")
    try:
        print(f"Enviando request na região `{nome}`")
        res = obj.request()
        for i in res:
            data.append(i)
    except Exception as e:
        print(f"Erro na região `{nome}`:{e}")
        erros+=1



print("------")
print(f"Erros:{erros}")


---------
Enviando request na região `BR_CS`
---------
Enviando request na região `US_CAL_CISO`
---------
Enviando request na região `FR`
Erro na região `FR`:nenhum registro com dado medido nos últimos 100
---------
Enviando request na região `Alemanha`
---------
Enviando request na região `GB`
---------
Enviando request na região `NZ`
---------
Enviando request na região `PL`
---------
Enviando request na região `ES`
---------
Enviando request na região `IT`
---------
Enviando request na região `SE`
------
Erros:1


In [43]:
df = pd.DataFrame(data, columns=["Região", "Horário", "Info"])
agora_sp = datetime.now(ZoneInfo("America/Sao_Paulo"))

df["Dia_Request"] = agora_sp.strftime("%d/%m/%Y")
df["Horário_Request"] = agora_sp.strftime("%H:%M")

expandido = pd.json_normalize(df["Info"])
df = df.drop(columns="Info").join(expandido)
df
# Dados_brutos.<fonte> é quanto cada fonte estava gerando naquele horário, em MW. 
# É o mix de geração "cru" que veio da API da região,
# antes de virar porcentagem ou intensidade.

,Região,Horário,Dia_Request,Horário_Request,total_mw,pct_renovavel,pct_livre_carbono,intensidade_gco2_kwh,dados_brutos.hidraulica,dados_brutos.termica_agregada,dados_brutos.eolica,dados_brutos.nuclear,dados_brutos.solar,dados_brutos.geotermica,dados_brutos.biomassa,dados_brutos.carvao,dados_brutos.gas,dados_brutos.oleo,dados_brutos.outros
0,Brasil SE/CO,2026-11-09T15:40:00-03:00,09/11/2026,15:56,40874.98,87.17,90.48,73.2,26687.43,3890.10,15.02,1355.24,8927.20,NaN,NaN,NaN,NaN,NaN,NaN
1,Brasil S,2026-11-09T15:40:00-03:00,09/11/2026,15:56,17895.79,91.90,91.90,66.8,12081.34,1449.33,374.48,NaN,3990.64,NaN,NaN,NaN,NaN,NaN,NaN
2,Brasil NE,2026-11-09T15:40:00-03:00,09/11/2026,15:56,23484.45,97.89,97.89,35.6,5069.28,495.90,10499.60,NaN,7419.67,NaN,NaN,NaN,NaN,NaN,NaN
3,Brasil N,2026-11-09T15:40:00-03:00,09/11/2026,15:56,7610.51,67.35,67.35,180.0,3392.15,2484.82,340.18,NaN,1393.36,NaN,NaN,NaN,NaN,NaN,NaN
4,EUA Califórnia,hoje 10:50 (horário do Pacífico),09/11/2026,15:56,29310.00,75.97,81.04,128.9,828.00,NaN,1591.00,1486.00,18862.00,688.0,298.0,2.0,5555.0,NaN,NaN
5,Alemanha,2026-11-09T17:15:00+00:00,09/11/2026,15:56,55530.70,28.12,28.12,491.8,6238.80,NaN,5019.20,NaN,NaN,13.2,4343.7,18511.8,19222.4,420.5,1761.1
6,Reino Unido,2026-11-09T18:00Z,09/11/2026,15:56,93.90,14.38,29.61,369.6,NaN,NaN,4.40,14.30,NaN,NaN,9.1,NaN,66.1,NaN,NaN
7,Nova Zelândia,10 Nov 2026 07:00 (horário NZ),09/11/2026,15:56,5446.00,95.56,95.56,53.8,3503.00,NaN,568.00,NaN,19.00,1114.0,NaN,97.0,102.0,NaN,43.0
8,Polônia,2026-11-09T16:00:00+00:00,09/11/2026,15:56,20802.10,11.08,11.08,674.7,990.30,NaN,951.50,NaN,NaN,NaN,362.3,14253.9,3485.4,177.9,580.8
9,Espanha,2026-11-09T16:15:00+00:00,09/11/2026,15:56,31764.00,54.84,73.98,145.2,3896.00,NaN,10740.00,6080.00,2332.00,NaN,452.0,NaN,7960.0,20.0,284.0


In [ ]:
s3 = S3Access()
s3.adicionar_dataframe("coletas.csv", df)